In [1]:
# import env vars
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
## make a client
from anthropic import Anthropic
client = Anthropic()
model ="claude-sonnet-4-0"

In [3]:
## user func
def add_user_messages(messages,text):
    user_message = {"role":"user","content":text}
    messages.append(user_message)

## assistant func
def add_assistant_messages(messages,text):
    assistant_message = {"role":"assistant","content":text}
    messages.append(assistant_message)

## chat func
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [8]:
import json
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, 
or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON,
 or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages=[]
    add_user_messages(messages,prompt)
    add_assistant_messages(messages,"```json")
    text=chat(messages,stop_sequences=["```"])
    return json.loads(text)

In [9]:
dataset=generate_dataset()
dataset

/var/folders/hl/xvs6dldn7nl02zjtlhj9vgbr0000gn/T/ipykernel_26412/4216836428.py:24: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


[{'task': "Write a Python function that extracts the AWS region from an S3 bucket ARN. The function should take an ARN string like 'arn:aws:s3:::my-bucket-us-west-2' and return the region code (e.g., 'us-west-2')."},
 {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'company-data-bucket'. The policy should include the necessary statements for listing and getting objects."},
 {'task': "Write a regex pattern that validates AWS EC2 instance IDs. The pattern should match strings that start with 'i-' followed by either 8 hexadecimal characters (older format) or 17 hexadecimal characters (newer format)."}]

In [10]:
## Write the output in the json
with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)